# Symbolic Execution with Guppy

This notebook demonstrates the symbolic stabilizer simulation workflow:

1. **Write quantum circuits in Guppy** - Python-embedded quantum DSL
2. **Compile to HUGR** - Hierarchical Unified Graph Representation
3. **Execute symbolically** - Track measurement dependencies without collapsing
4. **Sample efficiently** - Generate millions of shots in milliseconds!

## Why Symbolic Execution?

Traditional simulators collapse measurements to concrete outcomes. This means:
- Each shot requires a full simulation
- 1 million shots = 1 million simulations

Symbolic execution instead tracks *dependencies* between measurements:
- Execute once, capturing which measurements depend on which
- Sampling becomes XOR operations on random bits
- 1 million shots ≈ milliseconds!

In [ ]:
from guppylang import guppy
from guppylang.std.quantum import cx, h, measure, qubit
from pecos.experimental import execute_hugr_symbolic

## Example 1: Bell State

The simplest entangled state - two qubits that always measure the same.

In [ ]:
@guppy
def bell_state() -> tuple[bool, bool]:
    """Create and measure a Bell state: (|00⟩ + |11⟩)/√2."""
    q0 = qubit()
    q1 = qubit()
    h(q0)        # Put q0 in superposition
    cx(q0, q1)   # Entangle q0 and q1
    return (measure(q0), measure(q1))

# Compile to HUGR
package = bell_state.compile()
hugr_bytes = package.to_bytes()
print(f"HUGR size: {len(hugr_bytes)} bytes")

In [ ]:
# Execute symbolically
result = execute_hugr_symbolic(hugr_bytes)

print(f"Symbolic result: {result}")
print("\nMeasurement structure:")
print(f"  Total measurements: {result.num_measurements}")
print(f"  Deterministic: {result.num_deterministic}")
print(f"  Non-deterministic (random): {result.num_nondeterministic}")

The output `[m0=?, m1=m0]` tells us:
- `m0=?` - First measurement is random (50/50)
- `m1=m0` - Second measurement always equals the first

This is the Bell state correlation!

In [ ]:
# Sample 1 million shots - this is extremely fast!
import time

num_shots = 1_000_000

start = time.perf_counter()
counts = result.sample_counts(num_shots)
elapsed = time.perf_counter() - start

print(f"Generated {num_shots:,} shots in {elapsed*1000:.2f} ms")
print("\nOutcome counts:")
for outcome, count in sorted(counts.items()):
    bits = "".join(str(b) for b in outcome)
    print(f"  |{bits}⟩: {count:,} ({100*count/num_shots:.1f}%)")

## Example 2: GHZ State

A 3-qubit entangled state: (|000⟩ + |111⟩)/√2

In [ ]:
@guppy
def ghz_state() -> tuple[bool, bool, bool]:
    """Create and measure a 3-qubit GHZ state: (|000⟩ + |111⟩)/√2."""
    q0 = qubit()
    q1 = qubit()
    q2 = qubit()
    h(q0)
    cx(q0, q1)
    cx(q1, q2)
    return (measure(q0), measure(q1), measure(q2))

# Compile and execute symbolically
result = execute_hugr_symbolic(ghz_state.compile().to_bytes())

print(f"Symbolic result: {result}")
print("\nInterpretation:")
print("  m0=? : First qubit is random")
print("  m1=m0: Second qubit equals first")
print("  m2=m0: Third qubit equals first (via transitivity)")

In [ ]:
# Sample and verify
counts = result.sample_counts(100_000)

print("GHZ state outcomes (should only see 000 and 111):")
for outcome, count in sorted(counts.items()):
    bits = "".join(str(b) for b in outcome)
    print(f"  |{bits}⟩: {count:,}")

## Example 3: Repetition Code Syndrome Extraction

A more realistic example: 3-qubit repetition code with syndrome measurements.

```
Data qubits: q0, q1, q2 (encode logical qubit)
Ancillas: q3 (Z0Z1 check), q4 (Z1Z2 check)

Logical |+_L⟩ = (|000⟩ + |111⟩)/√2
```

In [ ]:
@guppy
def repetition_code_logical_plus() -> tuple[bool, bool, bool, bool, bool]:
    """3-qubit repetition code in logical |+_L⟩ state with syndrome extraction.

    Returns: (syndrome0, syndrome1, data0, data1, data2)
    """
    # Data qubits
    d0 = qubit()
    d1 = qubit()
    d2 = qubit()
    # Ancilla qubits for syndrome measurement
    a0 = qubit()
    a1 = qubit()

    # Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
    h(d0)
    cx(d0, d1)
    cx(d0, d2)

    # Syndrome extraction: Z0Z1 parity check
    # Uses CX gates to copy parity to ancilla
    cx(d0, a0)
    cx(d1, a0)
    s0 = measure(a0)

    # Syndrome extraction: Z1Z2 parity check
    cx(d1, a1)
    cx(d2, a1)
    s1 = measure(a1)

    # Measure data qubits
    m0 = measure(d0)
    m1 = measure(d1)
    m2 = measure(d2)

    return (s0, s1, m0, m1, m2)

# Execute symbolically
result = execute_hugr_symbolic(repetition_code_logical_plus.compile().to_bytes())

print(f"Symbolic result: {result}")
print("\nMeasurement structure:")
print(f"  Deterministic: {result.num_deterministic} (syndromes + correlated data)")
print(f"  Random: {result.num_nondeterministic} (logical qubit state)")

In [ ]:
# Sample and analyze
counts = result.sample_counts(100_000)

print("Repetition code outcomes (s0, s1, d0, d1, d2):")
print("Expected: syndromes=00, data qubits all same\n")

for outcome, count in sorted(counts.items(), key=lambda x: -x[1]):
    s0, s1, d0, d1, d2 = outcome
    syndrome = f"{s0}{s1}"
    data = f"{d0}{d1}{d2}"
    print(f"  syndrome={syndrome}, data={data}: {count:,}")

## Example 4: Comparing Sampling Performance

Let's compare the symbolic sampler against traditional simulation.

In [ ]:
import time

# Use the GHZ state for benchmarking
result = execute_hugr_symbolic(ghz_state.compile().to_bytes())

shot_counts = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

print("Symbolic sampling performance:")
print(f"{'Shots':>12} | {'Time (ms)':>10} | {'Shots/sec':>12}")
print("-" * 42)

for n in shot_counts:
    start = time.perf_counter()
    _ = result.sample_counts(n)
    elapsed = time.perf_counter() - start
    rate = n / elapsed
    print(f"{n:>12,} | {elapsed*1000:>10.2f} | {rate:>12,.0f}")

## How It Works: Measurement Dependencies

The symbolic simulator represents each measurement as:
- **Random bit index** (if non-deterministic) 
- **XOR of previous measurements** (dependencies)
- **Flip flag** (constant offset)

For example, in a Bell state:
- `m0 = random_bit[0]`
- `m1 = m0` (XOR of measurement 0, no flip)

Sampling just generates random bits and computes XORs - no matrix operations!

In [ ]:
# Get individual samples to see the structure
result = execute_hugr_symbolic(bell_state.compile().to_bytes())

print("First 10 samples from Bell state:")
samples = result.sample(10)
for i, sample in enumerate(samples):
    m0, m1 = sample
    print(f"  Shot {i}: m0={int(m0)}, m1={int(m1)} (equal: {m0 == m1})")

## Limitations

Symbolic execution works for **Clifford circuits** only:
- Supported gates: H, S, X, Y, Z, CX, CY, CZ, SWAP
- NOT supported: T gates, arbitrary rotations, non-Clifford gates

This is because Clifford circuits have efficient stabilizer representations,
while non-Clifford gates require exponential resources to simulate exactly.

In [ ]:
# This would fail with a non-Clifford gate:
# from guppylang.std.quantum import t
#
# @guppy
# def non_clifford():
#     q = qubit()
#     t(q)  # T gate is non-Clifford!
#     return measure(q)
#
# execute_hugr_symbolic(non_clifford.compile().to_bytes())
# RuntimeError: Unsupported gate for stabilizer simulation: T

## Summary

The `pecos.experimental` module provides:

- `execute_hugr_symbolic(hugr_bytes)` - Execute HUGR symbolically
- `execute_dag_circuit_symbolic(circuit)` - Execute DagCircuit directly
- `SymbolicExecutionResult` - Result with sampling methods:
  - `.sample(n)` - Get n samples as list of bool lists
  - `.sample_counts(n)` - Get n samples as outcome→count dict
  - `.num_measurements` - Total measurement count
  - `.num_deterministic` - Fixed/computed measurements
  - `.num_nondeterministic` - Random measurements

This enables extremely fast sampling for Clifford circuits, making it ideal for:
- Error correction simulations
- Syndrome extraction analysis
- Statistical sampling of stabilizer circuits